In [2]:
import vrep 
import sys
import time 
import numpy as np
from tank import *
import skfuzzy 
from skfuzzy import control as ctrl

In [21]:
def find_place(dist_EN, dist_ES):
    distance_ES = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'distance_ES')
    distance_EN = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'distance_EN')

    speed = ctrl.Consequent(np.arange(-1, 8.01, 0.01), 'speed')

    distance_EN['small'] = skfuzzy.trapmf(distance_EN.universe, [0, 0, 1.5, 1.55])
    distance_EN['average'] = skfuzzy.trapmf(distance_EN.universe, [1.5, 1.55, 2.5, 2.51])
    distance_EN['high'] = skfuzzy.trapmf(distance_EN.universe, [2.5, 2.51, 6, 6])

    # distance_ES['small'] = skfuzzy.trapmf(distance_ES.universe, [0, 0, 1.5, 1.55])
    # distance_ES['average'] = skfuzzy.trapmf(distance_ES.universe, [1.5, 1.55, 2.5, 2.51])
    # distance_ES['high'] = skfuzzy.trapmf(distance_ES.universe, [2.5, 2.51, 6, 6])

    distance_ES['small'] = skfuzzy.trapmf(distance_ES.universe, [0, 0, 1.1, 2])
    distance_ES['high'] = skfuzzy.trapmf(distance_ES.universe, [1.1, 2, 6, 6])

    speed['stop'] = skfuzzy.trimf(speed.universe, [-1, 0, 1])
    speed['break'] = skfuzzy.trimf(speed.universe, [0, 3, 5])
    speed['go'] = skfuzzy.trimf(speed.universe, [5, 6, 7])

    rules = (
        ctrl.Rule(distance_EN['high'] & distance_ES['high'], speed['go']),
        ctrl.Rule(distance_EN['small'] & distance_ES['small'], speed['go']),
        ctrl.Rule(distance_EN['small'] & distance_ES['high'], speed['go']),
        ctrl.Rule(distance_EN['high'] & distance_ES['small'], speed['break']),
        ctrl.Rule(distance_EN['average'] & distance_ES['high'], speed['break']),
        ctrl.Rule(distance_EN['average'] & distance_ES['small'], speed['stop']),
    )

    sim_ctrl = ctrl.ControlSystem(rules)
    sim = ctrl.ControlSystemSimulation(sim_ctrl)

    sim.input['distance_ES'] = dist_ES
    sim.input['distance_EN'] = dist_EN

    sim.compute()

    return sim.output['speed']

def set_to_park(dist_ES, dist_SE):
    distance_ES = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'distance_ES')
    distance_SE = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'distance_SE')

    speed = ctrl.Consequent(np.arange(-1, 8.01, 0.01), 'speed')

    distance_SE['poor'] = skfuzzy.trapmf(distance_SE.universe, [0, 0, 2.4, 3])
    distance_SE['good'] = skfuzzy.trapmf(distance_SE.universe, [2.4, 3, 6, 6])

    distance_ES['poor'] = skfuzzy.trapmf(distance_ES.universe, [0, 0, 1.1, 2])
    distance_ES['good'] = skfuzzy.trapmf(distance_ES.universe, [1.1, 2, 6, 6])

    speed['stop'] = skfuzzy.trimf(speed.universe, [-1, 0, 1])
    speed['break'] = skfuzzy.trimf(speed.universe, [0, 3, 5])
    speed['go'] = skfuzzy.trimf(speed.universe, [5, 6, 7])

    rules = (
        ctrl.Rule(distance_SE['good'] & distance_ES['good'], speed['go']),
        ctrl.Rule(distance_SE['good'] & distance_ES['poor'], speed['stop']),
        ctrl.Rule(distance_SE['poor'] & distance_ES['good'], speed['break']),
        ctrl.Rule(distance_SE['poor'] & distance_ES['poor'], speed['go'])
    )

    sim_ctrl = ctrl.ControlSystem(rules)
    sim = ctrl.ControlSystemSimulation(sim_ctrl)

    sim.input['distance_ES'] = dist_ES
    sim.input['distance_SE'] = dist_SE

    sim.compute()
    return sim.output['speed']

def drive_in_place(dist_ES, dist_SE):
    min_dist = min(dist_ES, dist_SE)

    car_dist = ctrl.Antecedent(np.arange(0, 6.01, 0.01), 'car_dist')
    speed = ctrl.Consequent(np.arange(-1, 8.01, 0.01), 'speed')

    car_dist['poor'] = skfuzzy.trapmf(car_dist.universe, [0, 0, 0.79, 0.80])
    car_dist['good'] = skfuzzy.trapmf(car_dist.universe, [0.79, 0.80, 6, 6])

    speed['stop'] = skfuzzy.trimf(speed.universe, [-1, 0, 1])
    speed['break'] = skfuzzy.trimf(speed.universe, [0, 1, 2])
    speed['go'] = skfuzzy.trimf(speed.universe, [5, 6, 7])

    rules = (
        ctrl.Rule(car_dist['good'], speed['go']),
        ctrl.Rule(car_dist['poor'], speed['stop']),
    )

    sim_ctrl = ctrl.ControlSystem(rules)
    sim = ctrl.ControlSystemSimulation(sim_ctrl)

    sim.input['car_dist'] = min_dist
    sim.compute()

    return sim.output['speed']

def parallel_park(dist_SW, dist_WS):
    min_dist = min(dist_SW, dist_WS)
    car_dist = ctrl.Antecedent(np.arange(0, 6, 0.01), 'car_dist')
    speed = ctrl.Consequent(np.arange(-1, 7, 0.01), 'speed')

    car_dist['poor'] = skfuzzy.trapmf(car_dist.universe, [0, 0, 1.1, 2])
    car_dist['good'] = skfuzzy.trapmf(car_dist.universe, [1.1, 2, 6, 6])

    speed['zero'] = skfuzzy.trimf(speed.universe, [-1, 0, 1])
    speed['poor'] = skfuzzy.trimf(speed.universe, [0, 1, 2])
    speed['good'] = skfuzzy.trimf(speed.universe, [5, 6, 7])


    rules = (
        ctrl.Rule(car_dist['good'], speed['good']),
        ctrl.Rule(car_dist['poor'], speed['zero']),
    )

    sim_ctrl = ctrl.ControlSystem(rules)
    sim = ctrl.ControlSystemSimulation(sim_ctrl)

    sim.input['car_dist'] = min_dist
    sim.compute()

    return sim.output['speed']


In [22]:
vrep.simxFinish(-1) # closes all opened connections, in case any prevoius wasnt finished
clientID=vrep.simxStart('127.0.0.1',19999,True,True,5000,5) # start a connection

if clientID!=-1:
    print ("Connected to remote API server")
else:
    print("Not connected to remote API server")
    sys.exit("Could not connect")

#create instance of Tank
tank=Tank(clientID)

Connected to remote API server


In [23]:
proximity_sensors=["EN","ES","NE","NW","SE","SW","WN","WS"]
proximity_sensors_handles=[0]*8

# get handle to proximity sensors
for i in range(len(proximity_sensors)):
    err_code,proximity_sensors_handles[i] = vrep.simxGetObjectHandle(clientID,"Proximity_sensor_"+proximity_sensors[i], vrep.simx_opmode_blocking)
    
#read and print values from proximity sensors
#first reading should be done with simx_opmode_streaming, further with simx_opmode_buffer parameter
for sensor_name, sensor_handle in zip(proximity_sensors,proximity_sensors_handles):
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,sensor_handle,vrep.simx_opmode_streaming)

In [24]:
tank.forward(5)

#continue reading and printing values from proximity sensors
distances = dict()
detection_states = dict()

for sensor_name, sensor_handle in zip(proximity_sensors,proximity_sensors_handles):
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,sensor_handle,vrep.simx_opmode_buffer)
        distances[sensor_name] = np.linalg.norm(detectedPoint)
        detection_states[sensor_name] = detectionState

stages = [
     'find_place',
     'set_to_park',
     'drive_in_place',
     'parallel_park'
]

t = time.time()
stage = 'find_place'
while (time.time()-t)<100: 
    for sensor_name, sensor_handle in zip(proximity_sensors,proximity_sensors_handles):
        err_code,detectionState,detectedPoint,detectedObjectHandle,detectedSurfaceNormalVector=vrep.simxReadProximitySensor(clientID,sensor_handle,vrep.simx_opmode_buffer )
        if(err_code == 0):
            # print("Proximity_sensor_"+sensor_name, np.linalg.norm(detectedPoint))
            distances[sensor_name] = np.linalg.norm(detectedPoint)
            detection_states[sensor_name] = detectionState
            for dist, state in zip(distances, detection_states):
                # print(f"{dist}, {detection_states[dist]}")
                pass
                 
            # if(detection_states['ES'] or detection_states['SE']):
            #     print(distances['SE'], distances['ES'])
            #     print(detection_states['SE'], detection_states['ES'])

    if stage == 'find_place':
        tank_speed = find_place(distances['EN'], distances['ES'])
        # print(tank_speed)
        if tank_speed < 2:
            tank.forward(0)
            stage = 'set_to_park'
        else:
            tank.forward(tank_speed)
    if stage == 'set_to_park':
        tank_speed = set_to_park(distances['ES'], distances['SE'])
        # print(tank_speed)
        if tank_speed < 1:
            tank.forward(0)
            stage = 'drive_in_place'
            print("Zmieniam na faze parkowania")
        else:
            tank.forward(tank_speed)
    if stage == 'drive_in_place':
        tank_speed = drive_in_place(distances['ES'],distances['SE'])
        # print(tank_speed)
        if tank_speed < 1:
            tank.forward(0)
            stage = 'parallel_park'
            print("Zmieniam faze na parallel")
        else:
            tank.leftvelocity = -tank_speed
            tank.rightvelocity = -tank_speed / 6
            tank.setVelocity()
    if stage == 'parallel_park':
        tank_speed = parallel_park(distances['SW'], distances['WS'])
        print(tank_speed)
        if tank_speed < 1:
            tank.stop()
            break
        else:
            tank.leftvelocity = -tank_speed / (2 + (1/3))
            tank.rightvelocity = -tank_speed
            tank.setVelocity()
    # print()

Zmieniam na faze parkowania
Zmieniam faze na parallel
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.999950330849877
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.961334124178064
5.88836799